In [1]:
import pandas as pd

In [2]:
import ee
import geemap

# ---- 1. Authenticate (first run opens a browser — sign in once) ----
ee.Authenticate()
ee.Initialize(project='my-first-project1-498804')   # <-- your GEE project id, see note below

# ---- 2. Load YOUR shapefile directly ----
shp_path = r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp"
lahore_fc = geemap.shp_to_ee(shp_path)
lahore = lahore_fc.geometry()

# ---- 3. VIIRS night-time lights: 2025 median, clipped to your boundary ----
ntl = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
       .filterDate('2025-01-01', '2026-01-01')
       .select('avg_rad')
       .median()
       .clip(lahore))

# ---- 4. Interactive map (shows in notebook) ----
Map = geemap.Map()
Map.centerObject(lahore, 10)
vis = {'min': 0, 'max': 60,
       'palette': ['000000', '3a2c5f', 'd97706', 'ffd166', 'ffffff']}
Map.addLayer(ntl, vis, 'NTL 2025')
Map.addLayer(ee.Image().paint(lahore, 1, 2), {'palette': ['E3A93C']}, 'District boundary')
Map

# ---- 5. Total radiance inside your district (number for your paper) ----
stats = ntl.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=lahore,
    scale=500,
    maxPixels=1e9
).getInfo()
print('Total radiance inside Lahore District:', stats)

# ---- 6. Download GeoTIFF straight to your PC ----
out_tif = r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif"
geemap.ee_export_image(
    ntl,
    filename=out_tif,
    scale=500,
    region=lahore,
    crs='EPSG:4326',
    file_per_band=False
)
print('Saved:', out_tif)

Total radiance inside Lahore District: {'avg_rad': 94585.8068709108}
Generating URL ...
Please wait ...
Data downloaded to E:\economic_wealth_dashboard\lahore_ntl_2025.tif
Saved: E:\economic_wealth_dashboard\lahore_ntl_2025.tif


In [3]:
import rasterio, json

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        cells.append([round(lng, 4), round(lat, 4), round(v, 2)])

json.dump(cells, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "lit cells written to ntl_grid.json")

17052 lit cells written to ntl_grid.json


In [4]:
import rasterio, numpy as np

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    print("pixel size (deg):", src.res)
    print("shape:", a.shape, "=", a.shape[0]*a.shape[1], "pixels")
    print("NaN pixels:", int(np.isnan(a).sum()))
    print("nodata value:", src.nodata)
    print("min:", np.nanmin(a), "max:", np.nanmax(a))
    print("finite pixels > 0.5:", int((np.nan_to_num(a) > 0.5).sum()))

pixel size (deg): (0.004491576420597608, 0.004491576420597608)
shape: (116, 147) = 17052 pixels
NaN pixels: 0
nodata value: None
min: 0.705 max: 86.895004
finite pixels > 0.5: 17052


In [5]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)          # top-left corner of the raster

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):   # keep ONLY inside your boundary
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district | px:", round(px, 6))

8426 cells inside district | px: 0.004492


In [6]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district")

8426 cells inside district


In [7]:
import osmnx as ox
import geopandas as gpd

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = shp.geometry.union_all()

tags = {"shop": True, "amenity": True, "office": True,
        "craft": True, "industrial": True, "tourism": True}

pois = ox.features_from_polygon(district, tags)
print("Total POIs:", len(pois))

Total POIs: 6545


In [8]:
print("Shops:", pois["shop"].notna().sum() if "shop" in pois else 0)
print("Amenities:", pois["amenity"].notna().sum() if "amenity" in pois else 0)
print("Offices:", pois["office"].notna().sum() if "office" in pois else 0)

print("\nTop 15 shop types:")
print(pois["shop"].value_counts().head(15))
print("\nTop 15 amenity types:")
print(pois["amenity"].value_counts().head(15))

Shops: 1137
Amenities: 4305
Offices: 567

Top 15 shop types:
shop
bakery              112
clothes              96
yes                  94
supermarket          91
mall                 84
convenience          43
car_repair           39
department_store     39
shoes                36
electronics          34
car                  34
books                22
mobile_phone         21
hairdresser          21
beauty               19
Name: count, dtype: int64

Top 15 amenity types:
amenity
place_of_worship    1110
restaurant           435
bank                 407
school               311
parking              249
hospital             226
fuel                 204
fast_food            142
cafe                  94
grave_yard            87
pharmacy              86
college               84
marketplace           79
university            72
fountain              72
Name: count, dtype: int64


In [9]:
print(pois["amenity"].value_counts().head(30))

amenity
place_of_worship    1110
restaurant           435
bank                 407
school               311
parking              249
hospital             226
fuel                 204
fast_food            142
cafe                  94
grave_yard            87
pharmacy              86
college               84
marketplace           79
university            72
fountain              72
clinic                59
atm                   58
police                54
community_centre      53
bus_station           40
events_venue          38
post_office           30
cinema                29
bench                 22
ice_cream             20
parking_entrance      18
dentist               17
doctors               17
shelter               15
smoking_area          15
Name: count, dtype: int64


In [12]:
keep = pois[["geometry"]].copy()
for col in ["shop", "amenity", "office", "craft", "tourism"]:
    keep[col] = pois[col] if col in pois else None
keep = keep[keep.geometry.notna()].to_crs(4326)
keep["lng"] = keep.geometry.centroid.x
keep["lat"] = keep.geometry.centroid.y
keep.drop(columns="geometry").to_csv(r"E:\economic_wealth_dashboard\lahore_pois.csv", index=False)
print("Saved lahore_pois.csv with", len(keep), "rows")

Saved lahore_pois.csv with 6545 rows


In [13]:
import pandas as pd

df = pd.read_csv(r"E:\economic_wealth_dashboard\lahore_pois.csv")

EXCLUDE = ["place_of_worship", "grave_yard", "fountain", "bench", "shelter",
           "police", "toilets", "drinking_water", "waste_basket", "recycling",
           "smoking_area", "post_box", "telephone", "waste_disposal"]
df = df[~df["amenity"].isin(EXCLUDE)]

SECTOR_MAP = {
  "Retail & Wholesale": {
    "shop": ["*"],
    "amenity": ["marketplace", "vending_machine"]},
  "Food & Hospitality": {
    "amenity": ["restaurant", "cafe", "fast_food", "food_court", "ice_cream", "juice_bar"],
    "tourism": ["hotel", "guest_house", "motel"]},
  "Financial & Professional": {
    "amenity": ["bank", "atm", "bureau_de_change", "money_transfer"],
    "office": ["*"]},
  "Transport & Logistics": {
    "amenity": ["fuel", "parking", "bus_station", "taxi", "car_wash", "car_rental"]},
  "Manufacturing": {
    "craft": ["*"], "industrial": ["*"]},
  "Health & Education": {
    "amenity": ["hospital", "clinic", "pharmacy", "doctors", "dentist",
                "school", "college", "university", "kindergarten", "library"]},
}

def classify(row):
    for sector, rules in SECTOR_MAP.items():
        for col, vals in rules.items():
            v = row.get(col)
            if pd.notna(v) and (vals == ["*"] or v in vals):
                return sector
    return "Other Services"

df["sector"] = df.apply(classify, axis=1)
counts = df["sector"].value_counts()
print("Economic POIs total:", len(df))
print(counts)
print("\nShares (%):")
print((100 * counts / counts.sum()).round(1))
df.to_csv(r"E:\economic_wealth_dashboard\lahore_pois_classified.csv", index=False)

Economic POIs total: 5154
sector
Retail & Wholesale          1218
Financial & Professional    1042
Health & Education           876
Food & Hospitality           870
Other Services               606
Transport & Logistics        511
Manufacturing                 31
Name: count, dtype: int64

Shares (%):
sector
Retail & Wholesale          23.6
Financial & Professional    20.2
Health & Education          17.0
Food & Hospitality          16.9
Other Services              11.8
Transport & Logistics        9.9
Manufacturing                0.6
Name: count, dtype: float64


In [14]:
import livepopulartimes

tests = [
    "Liberty Market, Gulberg, Lahore",
    "Bundu Khan Restaurant, MM Alam Road, Lahore",
    "HBL Main Boulevard Gulberg, Lahore",
]

for t in tests:
    try:
        r = livepopulartimes.get_populartimes_by_address(t)
        pt = r.get("populartimes")
        print("\n=== ", t, " ===")
        if pt:
            mon = next(d["data"] for d in pt if d["name"] == "Monday")
            print("Monday 24h:", mon)
        else:
            print("populartimes nahi mila (venue mila:", r.get("name"), ")")
    except Exception as e:
        print(t, "-> ERROR:", type(e).__name__, str(e)[:120])


===  Liberty Market, Gulberg, Lahore  ===
populartimes nahi mila (venue mila: Liberty Market )

===  Bundu Khan Restaurant, MM Alam Road, Lahore  ===
populartimes nahi mila (venue mila: None )

===  HBL Main Boulevard Gulberg, Lahore  ===
populartimes nahi mila (venue mila: None )


In [15]:
import livepopulartimes, json

tests = [
    "Packages Mall, Lahore, Pakistan",
    "Emporium Mall, Johar Town, Lahore, Pakistan",
    "McDonald's Gulberg, Lahore, Pakistan",
]

for t in tests:
    try:
        r = livepopulartimes.get_populartimes_by_address(t)
        print("\n===", t, "===")
        print("keys:", list(r.keys()))
        print("name:", r.get("name"), "| rating:", r.get("rating"))
        pt = r.get("populartimes")
        if pt:
            mon = next(d["data"] for d in pt if d["name"] == "Monday")
            print("Monday 24h:", mon)
        else:
            print("populartimes: EMPTY")
    except Exception as e:
        print(t, "-> ERROR:", type(e).__name__, str(e)[:150])


=== Packages Mall, Lahore, Pakistan ===
keys: ['rating', 'name', 'place_id', 'address', 'coordinates', 'categories', 'place_types', 'current_popularity', 'popular_times']
name: Packages Mall | rating: 4.6
populartimes: EMPTY

=== Emporium Mall, Johar Town, Lahore, Pakistan ===
keys: ['rating', 'name', 'place_id', 'address', 'coordinates', 'categories', 'place_types', 'current_popularity', 'popular_times']
name: Emporium Mall | rating: 4.6
populartimes: EMPTY

=== McDonald's Gulberg, Lahore, Pakistan ===
keys: ['rating', 'name', 'place_id', 'address', 'coordinates', 'categories', 'place_types', 'current_popularity', 'popular_times']
name: McDonald's | rating: 4.3
populartimes: EMPTY


In [16]:
import livepopulartimes

r = livepopulartimes.get_populartimes_by_address("Packages Mall, Lahore, Pakistan")

print("current_popularity (abhi is waqt):", r.get("current_popularity"))
pt = r.get("popular_times")
print("popular_times type:", type(pt))

if pt:
    for day in pt:
        print(day)
else:
    print("popular_times bhi khali hai:", pt)

current_popularity (abhi is waqt): None
popular_times type: <class 'NoneType'>
popular_times bhi khali hai: None


In [17]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = shp.geometry.union_all()

tags = {"shop": True, "amenity": True, "office": True, "craft": True, "tourism": True}
pois = ox.features_from_polygon(district, tags)

has_hours = pois["opening_hours"].notna().sum() if "opening_hours" in pois else 0
print("POIs with opening_hours:", has_hours, "out of", len(pois))

if has_hours:
    oh = pois[pois["opening_hours"].notna()]
    cols = [c for c in ["shop", "amenity", "office", "opening_hours"] if c in oh.columns]
    sample = oh[cols].head(20)
    print(sample.to_string())
    # save for parsing
    keep = oh[cols].copy()
    keep["lng"] = oh.geometry.centroid.x
    keep["lat"] = oh.geometry.centroid.y
    keep.to_csv(r"E:\economic_wealth_dashboard\lahore_opening_hours.csv", index=False)
    print("\nSaved lahore_opening_hours.csv")

POIs with opening_hours: 454 out of 6534
                                shop      amenity   office                                          opening_hours
element id                                                                                                       
node    300856816               mall          NaN      NaN                                             9am : 11pm
        1357561220               NaN     hospital      NaN                                                   24/7
        1715928613        baby_goods          NaN      NaN                                      Mo-Su 09:00-21:00
        1750454167               NaN      dentist      NaN                                      Mo-Fr 09:00-18:00
        2296900506  department_store          NaN      NaN                                            07:00-21:30
        2442864794           clothes          NaN      NaN                                      Mo-Sa 13:00-22:00
        2448135095           clothes          N

In [18]:
import pandas as pd, re, numpy as np

df = pd.read_csv(r"E:\economic_wealth_dashboard\lahore_opening_hours.csv")

# ── wohi sector classification jo pehle use ki ──
SECTOR_MAP = {
  "Retail & Wholesale": {"shop": ["*"], "amenity": ["marketplace"]},
  "Food & Hospitality": {"amenity": ["restaurant","cafe","fast_food","food_court","ice_cream"], "tourism": ["hotel","guest_house"]},
  "Financial & Professional": {"amenity": ["bank","atm","bureau_de_change"], "office": ["*"]},
  "Transport & Logistics": {"amenity": ["fuel","parking","bus_station","taxi","car_wash"]},
  "Health & Education": {"amenity": ["hospital","clinic","pharmacy","doctors","dentist","school","college","university"]},
}
def classify(row):
    for sec, rules in SECTOR_MAP.items():
        for col, vals in rules.items():
            v = row.get(col)
            if pd.notna(v) and (vals == ["*"] or v in vals): return sec
    return "Other Services"
df["sector"] = df.apply(classify, axis=1)

# ── opening_hours parser: har venue ke liye 24 flags (open=1/closed=0) ──
def to_hour(h, m, ampm=None):
    h = int(h)
    if ampm:
        ampm = ampm.lower()
        if ampm == "pm" and h != 12: h += 12
        if ampm == "am" and h == 12: h = 0
    return h % 24

def parse_hours(s):
    s = str(s).strip()
    if "24/7" in s: return np.ones(24)
    flags = np.zeros(24)
    # patterns: 09:00-17:00  |  9am-11pm  |  9am : 11pm
    pats = re.findall(r'(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*(?:-|:|till|to)\s*(\d{1,2})(?::(\d{2}))?\s*(am|pm)?', s, re.I)
    for h1, m1, ap1, h2, m2, ap2 in pats:
        try:
            a = to_hour(h1, m1 or 0, ap1); b = to_hour(h2, m2 or 0, ap2)
        except: continue
        if a == b: continue
        h = a
        while h != b:
            flags[h] = 1; h = (h + 1) % 24
    return flags if flags.sum() > 0 else None

parsed = df["opening_hours"].apply(parse_hours)
ok = parsed.notna()
print("Parsed successfully:", ok.sum(), "of", len(df))

# ── sector-wise average: har ghante kitne % venues khule ──
curves = {}
for sec in df["sector"].unique():
    arrs = [p for p, s, o in zip(parsed, df["sector"], ok) if o and s == sec]
    if len(arrs) >= 8:
        curves[sec] = np.mean(arrs, axis=0).round(3)
        print(f"\n{sec}  (n={len(arrs)})")
        print("  ", list(curves[sec]))

import json
json.dump({k: v.tolist() for k, v in curves.items()},
          open(r"E:\economic_wealth_dashboard\sector_open_curves.json", "w"))
print("\nSaved sector_open_curves.json")

Parsed successfully: 448 of 454

Retail & Wholesale  (n=142)
   [np.float64(0.155), np.float64(0.148), np.float64(0.155), np.float64(0.148), np.float64(0.148), np.float64(0.148), np.float64(0.183), np.float64(0.218), np.float64(0.352), np.float64(0.599), np.float64(0.838), np.float64(0.951), np.float64(0.972), np.float64(0.979), np.float64(0.979), np.float64(0.979), np.float64(0.979), np.float64(0.923), np.float64(0.838), np.float64(0.803), np.float64(0.676), np.float64(0.556), np.float64(0.317), np.float64(0.211)]

Health & Education  (n=56)
   [np.float64(0.321), np.float64(0.304), np.float64(0.268), np.float64(0.268), np.float64(0.268), np.float64(0.286), np.float64(0.339), np.float64(0.429), np.float64(0.536), np.float64(0.661), np.float64(0.857), np.float64(0.875), np.float64(0.875), np.float64(0.857), np.float64(0.839), np.float64(0.804), np.float64(0.786), np.float64(0.768), np.float64(0.732), np.float64(0.661), np.float64(0.643), np.float64(0.518), np.float64(0.411), np.float64

In [ ]:
import pandas as pd, json

df = pd.read_csv(r"E:\economic_wealth_dashboard\lahore_pois_classified.csv")

sectors = ["Retail & Wholesale","Food & Hospitality","Financial & Professional",
           "Transport & Logistics","Manufacturing","Health & Education","Other Services"]
sec_idx = {s: i for i, s in enumerate(sectors)}

def kind_of(row):
    for c in ["amenity","shop","office","craft","tourism"]:
        v = row.get(c)
        if pd.notna(v) and str(v) != "yes": return str(v)
    return "other"

pois = []
for _, r in df.iterrows():
    if pd.isna(r.get("lng")) or pd.isna(r.get("lat")): continue
    pois.append([round(float(r["lng"]),4), round(float(r["lat"]),4),
                 sec_idx.get(r["sector"], 6), kind_of(r)])

json.dump({"sectors": sectors, "pois": pois},
          open(r"E:\economic_wealth_dashboard\pois_data.json","w"))
print(len(pois), "POIs saved to pois_data.json")

5154 POIs saved to pois_data.json


In [2]:
json_path = r"E:\economic_wealth_dashboard\pois_data.json"
jsx_path  = r"E:\economic_wealth_dashboard\dashboard\src\EconomicPulse.jsx"
data = open(json_path, encoding="utf-8").read().strip()
src  = open(jsx_path,  encoding="utf-8").read()
if "PASTE_POIS_HERE" not in src:
    print("PASTE_POIS_HERE nahi mila")
else:
    open(jsx_path, "w", encoding="utf-8").write(src.replace("PASTE_POIS_HERE", data))
    print("POIs paste ho gaye! size:", len(data))

PASTE_POIS_HERE nahi mila
